In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
NUM_SAMPLES = 10000
SEQUENCE_LENGTH = 4
VOCAB_SIZE = 10

X = np.random.randint(
    1, VOCAB_SIZE + 1,
    size=(NUM_SAMPLES, SEQUENCE_LENGTH)
)

Y = X[:, ::-1]

# Decoder input starts with token 0
decoder_input = np.zeros_like(Y)

decoder_input[:, 1:] = Y[:, :-1]

In [3]:
# Encoder
encoder_inputs = Input(shape=(SEQUENCE_LENGTH,))

encoder_embedding = Embedding(
    VOCAB_SIZE + 1, 64
)(encoder_inputs)

encoder_lstm = LSTM(
    128, return_state=True
)

_, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(SEQUENCE_LENGTH,))

decoder_embedding = Embedding(
    VOCAB_SIZE + 1, 64
)(decoder_inputs)

decoder_lstm = LSTM(
    128, return_sequences=True
)

decoder_outputs = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

output_layer = Dense(
    VOCAB_SIZE + 1,
    activation="softmax"
)

decoder_outputs = output_layer(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4, 64)     │        704 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 4, 64)     │        704 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     98,816 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 4, 128)    │     98,816 │ embedding_1[0][0… │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4, 11)     │      1,419 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 200,459 (783.04 KB)

 Trainable params: 200,459 (783.04 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.fit(
    [X, decoder_input],
    Y,
    epochs=20,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.4737 - loss: 1.4760 - val_accuracy: 0.8978 - val_loss: 0.4964
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.9865 - loss: 0.1547 - val_accuracy: 0.9995 - val_loss: 0.0517
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9995 - loss: 0.0328 - val_accuracy: 0.9998 - val_loss: 0.0217
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 1.0000 - loss: 0.0155 - val_accuracy: 1.0000 - val_loss: 0.0109
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 1.0000 - loss: 0.0083 - val_accuracy: 1.0000 - val_loss: 0.0071
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 1.0000 - loss: 0.0055 - val_accuracy: 1.0000 - val_loss: 0.0048
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 0.0036
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accu

In [5]:
def reverse_sequence(sequence):

    decoder_seq = np.zeros(
        (1, SEQUENCE_LENGTH),
        dtype=int
    )

    result = []

    for i in range(SEQUENCE_LENGTH):

        prediction = model.predict(
            [np.array([sequence]), decoder_seq],
            verbose=0
        )

        next_token = np.argmax(
            prediction[0, i]
        )

        result.append(int(next_token))

        if i + 1 < SEQUENCE_LENGTH:
            decoder_seq[0, i + 1] = next_token

    return result


test_input = [1, 2, 3, 4]

print("Input Sequence:", test_input)
print("Reversed Sequence:", reverse_sequence(test_input))

Input Sequence: [1, 2, 3, 4]
Reversed Sequence: [4, 3, 2, 1]
